# Predicción de recuperación de oro
**Daniel Rodríguez · Ingeniería y Ciencia de Datos**

Caso académico de TripleTen. Copia de presentación con metodología revisada.

**Estado:** faltan los CSV para ejecutar el análisis y validar resultados. No se reutilizan las métricas originales como desempeño validado.

## 1. Objetivo
Estimar la recuperación de oro en flotación (rougher) y en el resultado final, usando predictores disponibles en el conjunto de prueba. Comparar una referencia simple y tres familias de regresión con sMAPE ponderado: 25 % rougher y 75 % final.

No se atribuyen resultados de producción ni mejoras operativas.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from analisis_oro import (read_table, prepare_training, compare_models, evaluate_reserved,
                         recovery_diagnostic, TARGETS)
DATA_DIR = Path('data')
train = read_table(DATA_DIR / 'gold_recovery_train.csv')
test = read_table(DATA_DIR / 'gold_recovery_test.csv')
if train.date.isin(test.date).any():
    raise ValueError('Revisar solapamiento temporal entre entrenamiento y prueba.')
display(pd.DataFrame({'filas':[len(train),len(test)],
                      'columnas':[train.shape[1],test.shape[1]]},index=['Train','Test']))
display(train.isna().sum().sort_values(ascending=False).head(15).to_frame('Ausentes train'))
print('Periodo train:',train.date.min(),train.date.max())
print('Periodo test:',test.date.min(),test.date.max())

## 2. Comprobación de la fórmula de recuperación
Se compara la fórmula metalúrgica con el objetivo original **antes de imputar**. Se excluyen de esta comparación denominadores nulos, valores no finitos y casos que no cumplan C ≥ F ≥ T ≥ 0. Estas exclusiones son diagnósticas; no se trasladan automáticamente al entrenamiento ni a la prueba. No se añade epsilon para transformar un cálculo indefinido en un resultado artificial.

In [ ]:
display(pd.Series(recovery_diagnostic(train),name='Diagnóstico de fórmula').to_frame())

## 3. Exploración del proceso
Visualizar concentraciones Au, Ag y Pb por etapa usando valores originales de train. Comparar la alimentación de train y test en unidades sin escalar mediante densidades normalizadas; una similitud visual no demuestra generalización. El conjunto test no se usa para elegir hiperparámetros ni reglas de depuración.

In [ ]:
stages = {'Alimentación':'rougher.input.feed_',
          'Flotación':'rougher.output.concentrate_',
          'Limpieza primaria':'primary_cleaner.output.concentrate_',
          'Final':'final.output.concentrate_'}
concentrations = pd.DataFrame({stage:{metal:train[prefix+metal].mean()
                              for metal in ['au','ag','pb']} for stage,prefix in stages.items()}).T
concentrations.plot(marker='o',figsize=(10,4),title='Concentración media por etapa (train)',ylabel='Concentración (unidad del dataset)')
plt.tight_layout(); plt.show()
column = 'rougher.input.feed_size'
fig,ax=plt.subplots(figsize=(9,4))
for label,df in [('Train',train),('Test',test)]:
    ax.hist(df[column].dropna(),bins=40,density=True,alpha=.5,label=label)
ax.set(xlabel='Tamaño de alimentación (unidad del dataset)',ylabel='Densidad',title='Distribución del tamaño de alimentación')
ax.legend(); plt.show()

## 4. Predictores y tratamiento de ausentes
Usar las columnas presentes en test, excluyendo date. Detenerse si el esquema incluye objetivos o variables de salida/cálculo. Confirmar también la disponibilidad de sensores en el momento de predicción.

No imputar objetivos. Informar cuántas filas no pueden usarse por falta de etiquetas válidas. El preprocesamiento de predictores se aprende dentro de cada partición de entrenamiento; no se filtra test por IQR.

In [ ]:
X,y,features,coverage = prepare_training(train,test)
display(pd.Series(coverage,name='Cobertura').to_frame())
print('Número de predictores:',len(features))
display(pd.Series(features,name='Predictores'))

## 5. Comparación con validación temporal
Referencia de la media, regresión lineal, árbol y bosque aleatorio. Pipelines con imputación mediana y escalado dentro de cada partición. Se conserva el orden temporal; las configuraciones de árbol/bosque son propuestas de partida, no parámetros optimizados.

La desviación entre particiones no es un intervalo de confianza. Revisar autocorrelación, horizonte y separación temporal necesaria antes de interpretar el resultado.

In [ ]:
models,cv_scores = compare_models(X,y,n_splits=5)
display(cv_scores)
cv_scores['sMAPE_CV'].plot.barh(color='#087f8c',xlabel='sMAPE ponderado (%)',title='Validación temporal: menor es mejor')
plt.tight_layout(); plt.show()
selected_name = cv_scores.index[0]
print('Candidato seleccionado solo con train:',selected_name)

## 6. Prueba reservada
Ejecutar una vez, después de fijar el modelo con la validación. Asociar objetivos desde full por fecha, sin incorporar sus otras columnas al entrenamiento. Evaluar el modelo elegido y la referencia sobre la misma cobertura. Si la referencia gana, comunicarlo.

No cambiar hiperparámetros usando este resultado. No eliminar valores extremos de prueba para mejorar la métrica.

In [ ]:
full = read_table(DATA_DIR / 'gold_recovery_full.csv')
final_metrics,predictions = evaluate_reserved(models[selected_name],X,y,test,full,features)
comparison = {selected_name:final_metrics}
if selected_name != 'Referencia (media)':
    baseline_metrics,_ = evaluate_reserved(models['Referencia (media)'],X,y,test,full,features)
    comparison['Referencia (media)'] = baseline_metrics
display(pd.DataFrame(comparison).T)
display(predictions.head())

## 7. Interpretación y próximos pasos
Las conclusiones cuantitativas quedan pendientes de ejecución real. Antes de comunicar resultados:

- Verificar cobertura de etiquetas, fechas y disponibilidad temporal de variables.
- Comprobar si el modelo supera la referencia con la misma métrica y población.
- Analizar diferencias entre rougher y final, estabilidad temporal y predicciones fuera del rango físico.
- Documentar procedencia de datos, versiones y límites de aplicación.

Un menor error predictivo no demuestra aumento de recuperación ni ahorro industrial. Consulta `DOCUMENTACION_TECNICA.md` para la revisión del original y los límites. El original permanece intacto.